# Build Photo3 Weather Input from Lab Weather Files

This notebook loads APOGEE and HOBO weather files, aggregates both to a configurable timestep, joins them on time, and exports a Photo3-ready file with columns:

- `datetime`
- `Temperature`
- `Relative Humidity`
- `GHI`

Default mapping in this notebook:

- APOGEE `Channel B` -> `GHI`
- HOBO `RH` -> `Relative Humidity`
- HOBO `Temperature` -> `Temperature`

In [1]:
from pathlib import Path
import pandas as pd

In [12]:
# -----------------------------
# User parameters (edit here)
# -----------------------------
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "sample_data" else cwd
base_dir = project_root / "sample_data"
print(f"Project root: {project_root}")
print(f"Base directory: {base_dir}")
data_dir = base_dir / "Lab_Weather_Data"

apogee_file = data_dir / "APOGEE_RGH101_1MIN_2026-6-11 to 7-10.xlsx"
hobo_file = data_dir / "HOBO_RGH101-Hartzell_10MINS_2026_6-11 to 7-10.xlsx"

# Output file will be written in sample_data
output_file = base_dir / "Lab_Weather_Joined_Interp30.xlsx"

# Aggregation timestep in minutes (e.g., 30, 10, 60)
timestep_minutes = 30

# Join behavior: "inner" keeps overlap only, "outer" keeps full range
join_how = "inner"

# When join_how="outer", interpolate missing values after join
interpolate_after_join = False

# Resampling aggregation methods
# Typical: mean for meteorological timeseries
apogee_agg = "mean"
hobo_agg = "mean"

# Optional explicit datetime column names
# If None, the notebook will auto-detect a datetime-like column
apogee_datetime_col = None
hobo_datetime_col = None

# Measurement column names
apogee_ghi_col = " Channel B (units?)"
hobo_rh_col = "RH , %"
hobo_temp_col = "Temperature , °C"

# Data cleaning controls
clip_negative_apogee_to_zero = True

print(f"APOGEE file: {apogee_file}")
print(f"HOBO file:   {hobo_file}")
print(f"Output file: {output_file}")
print(f"Timestep: {timestep_minutes} min | join: {join_how}")
print(f"Clamp APOGEE negatives to zero: {clip_negative_apogee_to_zero}")

Project root: c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits
Base directory: c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits\sample_data
APOGEE file: c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits\sample_data\Lab_Weather_Data\APOGEE_RGH101_1MIN_2026-6-11 to 7-10.xlsx
HOBO file:   c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits\sample_data\Lab_Weather_Data\HOBO_RGH101-Hartzell_10MINS_2026_6-11 to 7-10.xlsx
Output file: c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits\sample_data\Lab_Weather_Joined_Interp30.xlsx
Timestep: 30 min | join: inner
Clamp APOGEE negatives to zero: True


In [13]:
def _resolve_datetime_column(df: pd.DataFrame, explicit_col: str | None, label: str) -> str:
    if explicit_col is not None:
        if explicit_col not in df.columns:
            raise KeyError(f"{label}: datetime column '{explicit_col}' not found.")
        return explicit_col

    candidates = [
        c for c in df.columns
        if any(k in str(c).lower() for k in ["date", "time", "timestamp", "datetime"])
    ]
    for c in candidates:
        parsed = pd.to_datetime(df[c], errors="coerce")
        if parsed.notna().mean() > 0.8:
            return c

    raise ValueError(
        f"{label}: could not auto-detect a datetime column. Set the explicit datetime column name in the parameter cell."
    )


def _resolve_value_column(df: pd.DataFrame, requested_col: str, label: str) -> str:
    if requested_col in df.columns:
        return requested_col

    # Fallback: case-insensitive + trimmed match to make header selection less fragile.
    normalized = {str(c).strip().lower(): c for c in df.columns}
    key = str(requested_col).strip().lower()
    if key in normalized:
        return normalized[key]

    raise KeyError(
        f"{label}: value column '{requested_col}' not found. Available columns: {list(df.columns)}"
    )


def _load_and_resample_single_col(
    file_path: Path,
    value_col: str,
    datetime_col: str | None,
    out_col: str,
    timestep_min: int,
    agg: str,
    label: str,
    clip_negative_to_zero: bool = False,
) -> pd.DataFrame:
    df = pd.read_excel(file_path, engine="openpyxl")

    dt_col = _resolve_datetime_column(df, datetime_col, label)
    value_col_resolved = _resolve_value_column(df, value_col, label)

    out = pd.DataFrame()
    out["datetime"] = pd.to_datetime(df[dt_col], errors="coerce")
    out[out_col] = pd.to_numeric(df[value_col_resolved], errors="coerce")

    if clip_negative_to_zero:
        out[out_col] = out[out_col].clip(lower=0)

    out = out.dropna(subset=["datetime"]).set_index("datetime").sort_index()

    rule = f"{int(timestep_min)}min"
    if agg == "mean":
        out = out.resample(rule).mean()
    elif agg == "sum":
        out = out.resample(rule).sum()
    elif agg == "first":
        out = out.resample(rule).first()
    elif agg == "last":
        out = out.resample(rule).last()
    else:
        raise ValueError(f"Unsupported aggregation '{agg}'. Use one of: mean, sum, first, last.")

    return out

In [14]:
apogee_ghi = _load_and_resample_single_col(
    file_path=apogee_file,
    value_col=apogee_ghi_col,
    datetime_col=apogee_datetime_col,
    out_col="GHI",
    timestep_min=timestep_minutes,
    agg=apogee_agg,
    label="APOGEE",
    clip_negative_to_zero=clip_negative_apogee_to_zero,
)

hobo_temp = _load_and_resample_single_col(
    file_path=hobo_file,
    value_col=hobo_temp_col,
    datetime_col=hobo_datetime_col,
    out_col="Temperature",
    timestep_min=timestep_minutes,
    agg=hobo_agg,
    label="HOBO (Temperature)",
)

hobo_rh = _load_and_resample_single_col(
    file_path=hobo_file,
    value_col=hobo_rh_col,
    datetime_col=hobo_datetime_col,
    out_col="Relative Humidity",
    timestep_min=timestep_minutes,
    agg=hobo_agg,
    label="HOBO (RH)",
)

weather = apogee_ghi.join([hobo_temp, hobo_rh], how=join_how)

if join_how == "outer" and interpolate_after_join:
    weather = weather.interpolate(method="time", limit_direction="both")

weather = weather[["Temperature", "Relative Humidity", "GHI"]]

print("Joined weather shape:", weather.shape)
print("Datetime range:", weather.index.min(), "to", weather.index.max())
print("Columns:", list(weather.columns))
weather.head()

Joined weather shape: (1100, 3)
Datetime range: 2026-06-17 18:30:00 to 2026-07-10 16:00:00
Columns: ['Temperature', 'Relative Humidity', 'GHI']


,Temperature,Relative Humidity,GHI
datetime,,,
2026-06-17 18:30:00,23.855057,50.991313,16.071837
2026-06-17 19:00:00,23.326397,51.653544,14.204033
2026-06-17 19:30:00,22.997407,52.444967,12.215863
2026-06-17 20:00:00,23.217276,53.485616,1.166940
2026-06-17 20:30:00,24.395470,54.554240,0.049567


In [15]:
# Preview missingness before export
missing_summary = weather.isna().sum()
print("Missing values by column:")
print(missing_summary)

Missing values by column:
Temperature          0
Relative Humidity    0
GHI                  0
dtype: int64


In [17]:
# Export final input file (datetime is required)
weather.reset_index().to_excel(output_file, index=False)

print(f"Saved Photo3 input file (with datetime): {output_file}")

Saved Photo3 input file (with datetime): c:\Users\Josh.Gottlieb\OneDrive - Geosyntec\Documents\SourceCode\Photo3-Plant-Salinity-Traits\sample_data\Lab_Weather_Joined_Interp30.xlsx


## Notes

- To change the model timestep, edit `timestep_minutes` and rerun all cells.
- If datetime parsing fails, set `apogee_datetime_col` and `hobo_datetime_col` explicitly.
- If column names differ from defaults, update `apogee_ghi_col`, `hobo_rh_col`, and `hobo_temp_col`.
- If you need complete coverage instead of overlap only, set `join_how = "outer"` and optionally `interpolate_after_join = True`.
- The exported file includes `datetime` and is intended as the final model input file.